# We will make a GPT (Generative Pretrained Transformer) here including all the requreied componenets
# 

In [1]:
#Specifying  GPT with Dict

GPT_Config_124M= {

    "vocab_size" : 50257, #Vocabulary size for GPT2
    "context_length" : 1024, # Context Length
    "emb_dim":768, # Embedding Dimension
    "n_heads":12, # 12 attention heads
    "n_layers":12, # Number of layers
    "drop_rate":0.1, #Dropout rate
    "qkv_bias": False #Query,key,value bias

}

In [ ]:
# Now a dummy class for the whole GPT architecture

import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self,cfg): # here cfg is the configuration we defined above, more compact way
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"]) # we define token embeddings here
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["emb_dim"]) # we define the pos embedding here
        self.drop_emb=nn.Dropout(cfg["drop_rate"]) # the dropout rate

        #now the placeholder for transformer block -> this is the actual stufff we did in previous ch, meaning
        # the sself attention stuff etc, but for now a placeholder 

        self.trf_block=nn.Sequential(
            *[TransformerBlock(cfg)
            for _ in range(cfg["n_layers"])]
        )

        self.final_norm= LayerNorm(cfg["emb_dim"])

        self.out_head=nn.Linear(cfg["emb_dim"],cfg["vocab_size"],bias=False)

    def forward(self,in_idx):
        batch_size,seq_len=in_idx.shape
        tok_emb=self.tok_emb(in_idx)
        pos_emb=self.pos_emb(torch.arange(seq_len,device=in_idx.device))

        x=tok_emb+pos_emb #embedded vector?
        x=self.drop_emb(x)
        x=self.trf_block(x)
        x=self.final_norm(x)

        logits=self.out_head(x)
        return logits
    

# classes transformer block and layernorm, placeholders

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self,x):
        return x #just return for now
    

class LayerNorm(nn.Module):
    def __init__(self,normalized_shape,eps=1e-5):
        super().__init__()

    def forward(self,x):
        return x    



In [ ]:
#Running the class and feeding it some input
# we will prepare 2 batches of inputs and utilize tiktoken from ch2

import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
batch=[]
txt1="Every effort moves you"
txt2="Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

print(batch)
batch=torch.stack(batch,dim=0)
print("stacked batch: ",batch)

#nOW Intiliazing a dummy 124M parameter GPT  model   instance and feeding it this batch

torch.manual_seed(123)
model=DummyGPTModel(GPT_Config_124M)
logits=model(batch)
print("Output shape: ", logits.shape)
print("Output: ",logits)


             

[tensor([6109, 3626, 6100,  345]), tensor([6109, 1110, 6622,  257])]
stacked batch:  tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
Output shape:  torch.Size([2, 4, 50257])
Output:  tensor([[[-0.9289,  0.2748, -0.7557,  ..., -1.6070,  0.2702, -0.5888],
         [-0.4476,  0.1726,  0.5354,  ..., -0.3932,  1.5285,  0.8557],
         [ 0.5680,  1.6053, -0.2155,  ...,  1.1624,  0.1380,  0.7425],
         [ 0.0447,  2.4787, -0.8843,  ...,  1.3219, -0.0864, -0.5856]],

        [[-1.5474, -0.0542, -1.0571,  ..., -1.8061, -0.4494, -0.6747],
         [-0.8422,  0.8243, -0.1098,  ..., -0.1434,  0.2079,  1.2046],
         [ 0.1355,  1.1858, -0.1453,  ...,  0.0869, -0.1590,  0.1552],
         [ 0.1666, -0.8138,  0.2307,  ...,  2.5035, -0.3055, -0.3083]]],
       grad_fn=<UnsafeViewBackward0>)


# D10 --> We are doing layer normalization

In [ ]:
# we need a unit variance where mean is 0 and variance is 1

# we will be recreating book example with 5 inputs and 6 outputs and negatives are replaced 
# by zero using the nn.reLU function, ReLU is an activation function and there are several like this

torch.manual_seed(123)
batch_example=torch.randn(2,5) # create 2 examples with 5 dimensions(tokens) each
layer= nn.Sequential(nn.Linear(5,6),nn.ReLU())
out=layer(batch_example)
print(out)


# The output is a 6 dimension output from 5 inputs, know we will need to apply normalization layer
# first checking the current mean and variance

mean = out.mean(dim=-1,keepdim=True)
var= out.var(dim=-1,keepdim=True)
print("Mean: ", mean)
print("Variance: ", var)



#Now we apply layer normalization to the outputs and check the mean and variance 
# we do this by subtracting the mean from output and dvifing by square root of variance (standard deviation)

out_norm=(out-mean)/torch.sqrt(var)
print("Normalized layer output: ",out_norm)

torch.set_printoptions(sci_mode=False)
mean = out_norm.mean(dim=-1,keepdim=True)
var= out_norm.var(dim=-1,keepdim=True)
print("Mean: ", mean)
print("Variance: ", var)


In [ ]:
# Now implementing a class to use in GPT model later

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps=1e-5 # to avoid division by zero errors
        self.scale=nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self,x):
        mean=x.mean(dim=-1,keepdim=True)
        var=x.var(dim=-1,keepdim=True,unbiased=False)
        norm_x=(x-mean)/torch.sqrt(var+self.eps)
        return self.scale*norm_x+self.shift
    

#checking the class
Layer_n=LayerNorm(emb_dim=5)
out_Layer_n=Layer_n(batch_example)
mean=out_Layer_n.mean(dim=-1,keepdim=True)
var=out_Layer_n.var(dim=-1,unbiased=False,keepdim=True)
print("Mean: ", mean)
print("Variance: ", var)



Mean:  tensor([[    -0.0000],
        [     0.0000]], grad_fn=<MeanBackward1>)
Variance:  tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


# now the GELU module, it is more performative and is smoother curve than reLU

In [ ]:
# Now thwe GELU class
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,x):
        return 0.5 * x * (1+ torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))* 
                                        (x+0.44715*torch.pow(x,3))))
        